In [1]:

import pandas as pd
import gseapy as gp
import matplotlib.pyplot as plt
import scanpy as sc
import numpy as np

import pickle



from pydeseq2.preprocessing import deseq2_norm

# Loading

In [2]:
data_annotated_path = "/home/bmb/haxx/working/ceisel_mumm/data/"
data = sc.read_h5ad(data_annotated_path + "full_annotations_leiden_new.h5ad")


In [3]:
gp.get_library_name(organism="danio rerio")

In [4]:
zf_wiki = gp.get_library(name='WikiPathways_2018', organism='danio rerio')
zf_kegg = gp.get_library(name='KEGG_2019', organism='danio rerio')
zf_gobp = gp.get_library(name='GO_Biological_Process_2018', organism='danio rerio')

In [5]:
death_signatures = {
    'parthanatos':['mapk1', 'aif1l', 'dlg4a', 'appa', 'grin2bb', 'appb', 'parp1', 'plat',
       'endog', 'mmp9', 'parga', 'sirt1', 'map2k1', 'dlg4b', 'dlg4b-1', 'il1b',
       'pargl'],
    'necroptosis':['ripk3', 'tradd', 'cylda', 'fadd', 'pgam5', 'igfbp2a', 'cyldl',
       'ripk1l', 'igfbp2b', 'tnfrsf1a', 'vdac2', 'vdac1', 'cyldb', 'rbck1',
       'dnm1l'],
    'apoptosis':['pik3r1', 'prkar1b', 'baxb', 'prkar1aa', 'baxa', 'baxa-1', 'prkacaa',
       'cycsb', 'prkacbb', 'pik3r2', 'ppp3r1b', 'prkacab', 'ikbkb', 'irak1',
       'akt1', 'ppp3cb', 'xiap', 'ppp3r1a', 'prkar1ab', 'bcl2l1', 'birc2',
       'ppp3ca', 'prkacba', 'pik3ca']
}

In [6]:
pickle.dump(zf_wiki,open("zf_wiki.pkl","wb"))
pickle.dump(zf_kegg,open("zf_kegg.pkl","wb"))
pickle.dump(zf_gobp,open("zf_gobp.pkl","wb"))


# Preprocessing

In [7]:
zf_combined = zf_wiki | zf_kegg | death_signatures


In [8]:
raws = np.array(data.layers["raw"].todense()+1)
normed_raws,size_factors = deseq2_norm(raws)

In [9]:
normed_counts = pd.DataFrame(normed_raws.T,index=data.var_names,columns=data.obs_names)
normed_counts[~np.isfinite(normed_counts)] = 0
lognorm_counts = np.log2(normed_counts)


# Analysis

In [ ]:
# Run with more threads next time, 4 is conservative, 45gb python 
# phenotype = [1 if x == "Mtz" else 0 for x in data.obs["exp_condition"]]

# from gseapy import gsea
# gs = gsea(
#     data=lognorm_counts,
#     gene_sets=zf_combined,
#     cls = phenotype, # cls=class_vector
#     # set permutation_type to phenotype if samples >=15
#     permutation_type='phenotype',
#     permutation_num=1000, # reduce number to speed up test
#     outdir=None,
#     method='signal_to_noise',
#     threads=10, seed= 8
# )
# gs_res = gs.run()

In [ ]:
# import pickle

# pickle.dump(gs,open("gs_lognorm_combined.pkl","wb"))

# pickle.dump(gs_res,open("gs_res.pkl","wb"))
# pickle.dump(gs,open("gs.pkl","wb"))

In [ ]:
gs = pickle.load(open("gs_lognorm_combined.pkl","rb"))

In [ ]:
terms = gs.res2d.Term
axs = gs.plot(terms[:5], show_ranking=False, legend_kws={'loc': (1.05, 0)}, )

In [11]:
# phenotype = [1 if x == "Mtz" else 0 for x in data.obs["exp_condition"]]


from gseapy import gsea
gs = gsea(
    data=normed_counts,
    gene_sets=zf_combined,
    cls = data.obs['exp_condition'],
    # set permutation_type to phenotype if samples >=15
    permutation_type='phenotype',
    permutation_num=1000, # reduce number to speed up test
    outdir=None,
    method='signal_to_noise',
    threads=10, seed= 8
)
gs.pheno_neg="Cntr"
gs.pheno_pos="Mtz"
_res = gs.run()

pickle.dump(gs,open("gs_denorm_combined.pkl","wb"))


In [10]:
gs = pickle.load(open("gs_denorm_combined.pkl","rb"))

In [11]:
# terms = gs.res2d.Term
# axs = gs.plot(terms[:5], show_ranking=False, legend_kws={'loc': (1.05, 0)}, )
gs.plot(['parthanatos','necroptosis','apoptosis'])

In [ ]:
phenotype = [1 if x == "Mtz" else 0 for x in data.obs["exp_condition"]]

from gseapy import gsea
gs = gsea(
    data=normed_counts,
    gene_sets=zf_gobp,
    cls = phenotype, # cls=class_vector
    # set permutation_type to phenotype if samples >=15
    permutation_type='phenotype',
    permutation_num=1000, # reduce number to speed up test
    outdir=None,
    method='signal_to_noise',
    threads=10, seed= 8
)
gs_res = gs.run()

pickle.dump(gs,open("gs_denorm_gobp.pkl","wb"))

In [ ]:
gs = pickle.load(open("gs_denorm_gobp.pkl","rb"))

In [ ]:
terms = gs.res2d.Term
axs = gs.plot(terms[:5], show_ranking=False, legend_kws={'loc': (1.05, 0)}, )

# Cell Type Specific

In [ ]:
# rgc_normed = normed_counts.loc[:,rgc_mask]
# rgc_phenotype = np.array(data.obs['exp_condition'])[rgc_mask]

# gs = gp.gsea(
#     data=rgc_normed,
#     gene_sets=zf_combined,
#     cls = rgc_phenotype, # cls=class_vector
#     # set permutation_type to phenotype if samples >=15
#     permutation_type='phenotype',
#     permutation_num=1000, # reduce number to speed up test
#     outdir=None,
#     method='signal_to_noise',
#     threads=10, seed= 8
# )
# gs.pheno_neg="Cntr"
# gs.pheno_pos="Mtz"
# gs_res = gs.run()

# terms = gs.res2d.Term
# axs = gs.plot(terms[:5], show_ranking=False, legend_kws={'loc': (1.05, 0)}, )

# pickle.dump(gs,open("gs_rgc_denorm_combined.pkl","wb"))

In [ ]:
# gs_bipolar,_ = run_cell_type(
#     data,
#     'Bipolar cells','bipolar',
#     zf_combined,'combined'
# )

# run_cell_type(
#     data,
#     'Bipolar cells','bipolar',
#     zf_gobp,'GOBP'
# )
# target_population = "Bipolar cells"
# target_label = "bipolar"


# target_mask = data.obs['cell_type'] == "Bipolar cells"
# print(np.sum(target_mask))
# target_normed = normed_counts.loc[:,target_mask]
# target_phenotype = np.array(data.obs['exp_condition'])[target_mask]

# gs = gp.gsea(
#     data=target_normed,
#     gene_sets=zf_combined,
#     cls = target_phenotype, # cls=class_vector
#     # set permutation_type to phenotype if samples >=15
#     permutation_type='phenotype',
#     permutation_num=1000, # reduce number to speed up test
#     outdir=None,
#     method='signal_to_noise',
#     threads=10, seed= 8
# )
# gs.pheno_neg="Cntr"
# gs.pheno_pos="Mtz"
# gs_res = gs.run()

# terms = gs.res2d.Term
# axs = gs.plot(terms[:5], show_ranking=False, legend_kws={'loc': (1.05, 0)}, )

# pickle.dump(gs,open(f"gs_{target_label}_denorm_combined.pkl","wb"))

In [13]:
# gs,axs = run_cell_type(
#     data,
#     'RGCs','rgc',
#     zf_combined,'combined'
# )

# gs,axs = run_cell_type(
#     data,
#     'RGCs','rgc',
#     zf_gobp,'GOBP'
# )

# gs_bipolar = pickle.load(open("gs_bipolar_denorm_combined.pkl","rb"))
gs_bipolar.plot(['parthanatos','necroptosis','apoptosis'])

In [ ]:
def run_cell_type(
    data,
    target_population = "Bipolar cells",
    target_label = "bipolar",
    gene_sets = zf_gobp,
    gene_set_label = "GOBP"
):

    target_mask = data.obs['cell_type'] == target_population
    target_normed = normed_counts.loc[:,target_mask]
    target_phenotype = np.array(data.obs['exp_condition'])[target_mask]

    print(f"{target_population} size: {np.sum(target_mask)}")
    print(f"Phenotype breakout: {np.unique(target_phenotype,return_counts=True)}")

    gs = gp.gsea(
        data=target_normed,
        gene_sets=gene_sets,
        cls = target_phenotype, # cls=class_vector
        # set permutation_type to phenotype if samples >=15
        permutation_type='phenotype',
        permutation_num=1000, # reduce number to speed up test
        outdir=None,
        method='signal_to_noise',
        threads=10, seed= 8
    )
    gs.pheno_neg="Cntr"
    gs.pheno_pos="Mtz"
    gs.run()

    terms = gs.res2d.Term
    axs = gs.plot(terms[:5], show_ranking=False, legend_kws={'loc': (1.05, 0)}, )
    

    pickle.dump(gs,open(f"gs_{target_label}_denorm_{gene_set_label}.pkl","wb"))
    
    return gs,axs

In [ ]:
# gs.plot(['parthanatos','necroptosis','apoptosis'])
gs_bipolar.plot(['parthanatos','necroptosis','apoptosis'])

In [ ]:
rgc_phenotype

In [ ]:
present = [gene for gene in data.var_names if gene in flat_kegg_set]
missing = [gene for gene in data.var_names if gene not in flat_kegg_set]
lower = [gene for gene in missing if gene.lower() in flat_kegg_set]
print(f"Present: {len(present)}, Missing: {len(missing)}")